**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Cyclostationarity & Higher-Order Statistics

Two escapes from the [WSS/Gaussian](./Statistical_Signal_Processing.ipynb) worldview: signals whose *statistics* repeat periodically (every modulated signal!), detectable even **below the noise floor** — and higher-order moments that see what covariance is blind to. This is how [SDR](../Intro_SDR/Software_Defined_Radio.ipynb) detectors find signals the PSD can't.

## 1. Pre-requisites

[Statistical SP](./Statistical_Signal_Processing.ipynb), [Digital Communications](./Digital_Communications.ipynb) S1–S2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Cyclostationarity: Rhythm in the Statistics* (~40 min)
**Goal:** see why modulated signals aren't WSS; meet the cyclic autocorrelation.
**Builds on:** [Statistical SP](./Statistical_Signal_Processing.ipynb) S1. &nbsp; **Feeds into:** Session 2 (detection below the noise floor).

---

## 2. Periodic Statistics

💡 **Intuition.** A BPSK signal's *mean power* pulses at the symbol rate — the signal is not WSS, it's **cyclostationary**: statistics periodic in time. Fourier-expand the time-varying autocorrelation and you get the **cyclic autocorrelation** $R_x^\alpha(\tau)$ at *cycle frequencies* α — nonzero exactly at multiples of the symbol rate (and around twice the carrier). White noise, being genuinely stationary, has **zero** cyclic content at any α ≠ 0 — and that asymmetry is an exploitable superpower.

In [ ]:
# a BPSK signal, its cyclic autocorrelation at candidate cycle frequencies

# YOUR CODE HERE


**What just happened.** The cyclic autocorrelation is essentially flat across candidate cycle frequencies except for spikes at two places: **250 Hz**, the symbol rate, and **2000 Hz**, twice the carrier. Both were marked in advance, and both are exactly where the theory says they must be.

**Why those two, specifically.** The symbol rate appears because the symbol timing is periodic — the signal's power pulses once per symbol, so its second-order statistics repeat at 250 Hz. Twice the carrier appears from the trigonometry: squaring a carrier gives $\cos^2(2\pi f_c t) = \tfrac12[1 + \cos(4\pi f_c t)]$, so a second-order statistic of a signal carried at $f_c$ contains a component at $2f_c$, never at $f_c$ itself. Students consistently expect a spike at 1000 Hz; the factor of two is a direct consequence of looking at a *quadratic* statistic.

**Note carefully what axis this is.** The horizontal axis is $\alpha$, the **cycle frequency** — the rate at which the signal's statistics repeat — not ordinary frequency. This is a second frequency axis describing the rhythm of the statistics rather than of the waveform. Conflating the two is the standard confusion in this subject; the PSD of this signal would show a hump around 1000 Hz and nothing whatsoever at 250 Hz.

**And here is the asymmetry that makes all of this useful.** White noise is genuinely stationary — its statistics do not vary with time at all — so its cyclic content at every $\alpha \neq 0$ is **exactly zero in expectation**. Not small: zero. A finite record produces only estimation noise, which shrinks as $1/\sqrt{N}$ with record length.

That is a difference in kind rather than degree, and it changes what is possible. An energy detector compares signal power against noise power, so it fails once the signal is weaker than the noise. A cyclic detector evaluates a quantity to which noise contributes *nothing* on average, so with enough record length the signal's rhythm can be pulled out from arbitrarily far below the noise floor. Session 2 measures exactly that.

**One practical framing.** These two spikes constitute a *fingerprint*: they reveal the symbol rate and carrier frequency of a signal nobody demodulated and whose parameters were never supplied. Blind modulation classification and spectrum sensing are built on reading this plot. Note also that `cyclic_acf` here is a deliberately literal Python loop; production sensors compute the whole spectral correlation surface with FFT-based algorithms (FAM, SSCA) fast enough to run live.

---
### 🕐 Session 2 of 3 — *Detection Below the Noise Floor* (~40 min)
**Goal:** energy detection dies at 0 dB; the cyclic detector keeps working well below it.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (higher-order statistics).

---

## 3. The Superpower, Cashed

💡 **Intuition.** An energy detector asks 'is total power above the noise baseline?' — hopeless when SNR < 0 dB and the noise level is uncertain. The **cyclic detector** asks 'is there energy at cycle frequency α = symbol rate?' — and since noise contributes *nothing* there (only estimation noise, shrinking with record length), the signal's rhythm shines through arbitrarily far below the floor, given time. This is how spectrum sensors find hidden transmitters.

In [ ]:
    # REALISM: the receiver's noise level is only known to ±1 dB (temperature, gain drift)

# YOUR CODE HERE


Read the table: the energy detector's separation collapses with SNR (and would collapse entirely under noise-level uncertainty), while the cyclic statistic keeps the classes many σ apart — the rhythm survives where the power story drowns.

---
### 🕐 Session 3 of 3 — *Higher-Order Statistics* (~35 min)
**Goal:** what covariance can't see: kurtosis, the bispectrum, and Gaussian blindness.
**Builds on:** Session 2.

---

## 4. Beyond Second Order

💡 **Intuition.** Covariance sees only the ellipse of a distribution. **Higher-order cumulants** see shape: kurtosis (4th) measures tail-heaviness — the compass [ICA](./ICA_Blind_Source_Separation.ipynb) steers by — and the **bispectrum** (the 2-D Fourier transform of the 3rd-order cumulant) detects *phase coupling*: components at $f_1, f_2, f_1{+}f_2$ with locked phases, the fingerprint of nonlinearity. The master fact: **all cumulants above 2nd order of a Gaussian are exactly zero** — so anything nonzero up there is, provably, structure.

In [ ]:
# quadratic phase coupling: visible to the bispectrum, invisible to the PSD
# phases drift slowly and independently between oscillators (physical: separate sources);
# in the COUPLED signal the third tone's phase is SLAVED to ph1+ph2 at every instant

# YOUR CODE HERE


**What just happened.** Bicoherence **0.977** for the coupled signal against **0.110** for the uncoupled one — from two signals whose power spectra are *identical*.

That identity is the point, so be precise about it. Both signals contain tones at $f_1 = 480$, $f_2 = 700$, and $f_1 + f_2 = 1180$ Hz with the same amplitudes. The PSD measures $|X(f)|^2$ and therefore discards phase entirely; it cannot express the difference between these two signals even in principle. What differs is a phase *relation*: in the coupled signal the third tone's phase is slaved to $\phi_1 + \phi_2$ at every instant, while in the uncoupled one it drifts independently. Second-order statistics are structurally blind to relations between different frequencies.

The bicoherence sees it because it averages the *triple product* $X(f_1)X(f_2)X^*(f_1{+}f_2)$ across segments. When the phases are locked, that product has a consistent phase in every segment and the terms add coherently — near 1. When the third phase drifts freely, the product's phase is random per segment and the sum cancels — near 0.

**Read the 0.110 correctly, because it is not zero.** With `nseg = 64` segments averaged, the expected bias floor for genuinely uncorrelated phases is roughly $1/\sqrt{64} = 0.125$. The measured 0.110 sits right at that floor, so it is **statistically indistinguishable from zero** — random phases cancel only as fast as $1/\sqrt{N}$, never exactly. This is the honest reading, and it matters: quoting 0.110 as "small" invites the question of how small is small, whereas quoting it against its own noise floor answers it. Increase `nseg` and the floor drops accordingly.

**Why this is worth caring about physically.** Quadratic phase coupling is the signature of a **nonlinearity**. Put two tones through any squaring element and the cross-term appears at $f_1 + f_2$ with phase necessarily equal to $\phi_1 + \phi_2$ — that is simply what multiplication does to phases. So detecting phase coupling is detecting that a nonlinear process generated the data, rather than three unrelated oscillators that happen to sit at arithmetically related frequencies. That distinction is invisible to the PSD and is exactly what you need when hunting distortion in an amplifier, coupled rhythms in EEG, or wave interactions in a plasma.

**The unifying fact behind the whole session.** All cumulants above second order of a Gaussian are **exactly zero**. So higher-order statistics ignore Gaussian noise for free, without needing to know its level, and anything nonzero up there is provably non-Gaussian structure. That is the same difference-in-kind that made Session 2's cyclic detector immune to the noise floor, reached by a different route — and it is why [ICA](./ICA_Blind_Source_Separation.ipynb) can use kurtosis as a compass for finding independent components.

**The cost, stated plainly.** Higher-order estimates are variance-hungry — a third- or fourth-order quantity needs far more data than a covariance for comparable precision — and they are sensitive to outliers. Reach for them when the structure you need is genuinely invisible at second order, which here it provably was.

## 5. Conclusion

Modulation puts rhythm into statistics; that rhythm detects signals the PSD loses (σ-separations measured); and third/fourth-order cumulants see phase coupling and non-Gaussian shape where covariance sees nothing at all. When the standard assumptions fail, these are the tools that notice.

---
## Where next

- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — cyclic detection on live captures.
- [ICA](./ICA_Blind_Source_Separation.ipynb) — kurtosis as a steering wheel.
- [Digital Communications](./Digital_Communications.ipynb) — the signals whose rhythms we exploited.